In [1]:
import sys
sys.path.append('..')  # if running from notebooks/


import pandas as pd
from src.data.database import get_raw_prices_df

raw_df = get_raw_prices_df(commodity='onion')
raw_df['date'] = pd.to_datetime(raw_df['date'])

lasalgaon = raw_df[raw_df['market'].str.contains('Lasalgaon', case=False)]
print(f"Lasalgaon records: {len(lasalgaon)}")
print(f"Date range: {lasalgaon['date'].min().date()} to {lasalgaon['date'].max().date()}")

print("\nTop 10 markets by records:")
print(raw_df['market'].value_counts().head(10))

Lasalgaon records: 1756
Date range: 2023-01-07 to 2025-12-05

Top 10 markets by records:
market
Kayamkulam                                   1193
Bangalore                                    1184
Hubli (Amaragol)                             1108
Kanjirappally                                1078
Pratapgarh                                   1005
Chengannur                                    972
Ahmedabad(Chimanbhai Patal Market Vasana)     915
Thalayolaparambu                              900
Thodupuzha                                    876
Mahuva(Station Road)                          864
Name: count, dtype: int64


In [4]:
import sys
sys.path.append('..')

import pandas as pd
from src.data.database import get_raw_prices_df
from src.features.engineer import FeatureEngineer

# Get all Lasalgaon records combined
raw_df = get_raw_prices_df(commodity='onion')
lasalgaon_df = raw_df[raw_df['market'].str.contains('Lasalgaon', case=False)].copy()
print(f"Total Lasalgaon records: {len(lasalgaon_df)}")
print(f"Date range: {lasalgaon_df['date'].min()} to {lasalgaon_df['date'].max()}")

# Save as a temporary CSV so engineer can pick it up
from src.utils.config import PROCESSED_DIR
import numpy as np

lasalgaon_df['date'] = pd.to_datetime(lasalgaon_df['date'])

# Daily average across the 3 Lasalgaon markets
daily = lasalgaon_df.groupby('date').agg(
    modal_price=('modal_price', 'mean'),
    min_price=('min_price', 'mean'),
    max_price=('max_price', 'mean'),
    n_markets=('market', 'nunique'),
).reset_index()

print(f"\nDaily rows: {len(daily)}")
print(f"Missing dates: {(pd.date_range(daily['date'].min(), daily['date'].max()).difference(daily['date'])).shape[0]}")
print(daily.tail())

Total Lasalgaon records: 1756
Date range: 2023-01-07 00:00:00 to 2025-12-05 00:00:00

Daily rows: 524
Missing dates: 540
          date  modal_price    min_price    max_price  n_markets
519 2025-11-06  1615.000000   525.500000  1921.000000          2
520 2025-12-02  2675.000000  1150.000000  3101.500000          2
521 2025-12-03  1492.500000   916.666667  1767.166667          3
522 2025-12-04  1025.000000   603.333333  1133.833333          3
523 2025-12-05  1131.666667   500.000000  1415.000000          3


In [5]:
import sys
sys.path.append('..')
import pandas as pd

df = pd.read_csv('../data/processed/features_onion.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

n = len(df)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)
n_test  = n - n_train - n_val

print(f"Total rows:  {n}")
print(f"Train:       {n_train}")
print(f"Val:         {n_val}")
print(f"Test:        {n_test}")
print(f"Lookback:    60")
print(f"Usable train sequences for LSTM: {n_train - 60}")
print(f"\nPrice stats:")
print(df['modal_price'].describe().round(2))
print(f"\nNull counts in key features:")
print(df[['modal_price','lag_7d','rolling_mean_30d','temp_max','precip']].isnull().sum())

Total rows:  1034
Train:       723
Val:         155
Test:        156
Lookback:    60
Usable train sequences for LSTM: 663

Price stats:
count    1034.00
mean     2652.08
std      1108.90
min      1197.44
25%      1766.08
50%      2114.25
75%      3721.62
max      5303.51
Name: modal_price, dtype: float64

Null counts in key features:
modal_price         0
lag_7d              0
rolling_mean_30d    0
temp_max            0
precip              0
dtype: int64


In [6]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/features_onion.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

# Check target variable
print("=== TARGET VARIABLE (target_1d) ===")
print(df['target_1d'].describe().round(2))
print(f"Nulls: {df['target_1d'].isnull().sum()}")

# Check if target_1d makes sense
print("\n=== SAMPLE: date, modal_price, target_1d ===")
print(df[['date','modal_price','target_1d']].head(10).to_string())

# Check correlation between features and target
print("\n=== TOP CORRELATIONS WITH target_1d ===")
numeric = df.select_dtypes(include=[np.number])
corr = numeric.corr()['target_1d'].abs().sort_values(ascending=False)
print(corr.head(15))

# Check train/test price ranges
n = len(df)
n_train = int(n * 0.70)
n_val = int(n * 0.15)
train = df.iloc[:n_train]
test  = df.iloc[n_train+n_val:]
print(f"\n=== PRICE RANGES ===")
print(f"Train mean: ₹{train['modal_price'].mean():.0f} | std: ₹{train['modal_price'].std():.0f}")
print(f"Test mean:  ₹{test['modal_price'].mean():.0f} | std: ₹{test['modal_price'].std():.0f}")

=== TARGET VARIABLE (target_1d) ===
count    1033.00
mean     2650.91
std      1108.79
min      1197.44
25%      1766.08
50%      2113.09
75%      3721.62
max      5303.51
Name: target_1d, dtype: float64
Nulls: 1

=== SAMPLE: date, modal_price, target_1d ===
        date  modal_price    target_1d
0 2023-02-06  3860.242507  1419.551471
1 2023-02-07  1419.551471  1554.813043
2 2023-02-08  1554.813043  2145.933174
3 2023-02-09  2145.933174  2159.978056
4 2023-02-10  2159.978056  4226.404922
5 2023-02-11  4226.404922  3821.023018
6 2023-02-12  3821.023018  3821.023018
7 2023-02-13  3821.023018  3821.023018
8 2023-02-14  3821.023018  3821.023018
9 2023-02-15  3821.023018  3821.023018

=== TOP CORRELATIONS WITH target_1d ===
target_1d            1.000000
target_2d            0.906175
modal_price          0.903940
min_price            0.901278
max_price            0.900335
target_3d            0.800795
lag_1d               0.796804
price_spread         0.752815
rolling_mean_7d      0.733685
t

In [7]:
# Check how many rows are forward-filled (duplicates in modal_price)
consecutive_same = (df['modal_price'] == df['modal_price'].shift(1)).sum()
print(f"Consecutive same-price rows (forward-filled): {consecutive_same}")
print(f"That's {consecutive_same/len(df)*100:.1f}% of all rows")

# Check test period dates
print(f"\nTest period: {test['date'].min().date()} to {test['date'].max().date()}")
print(f"Train period: {train['date'].min().date()} to {train['date'].max().date()}")

# Price by year
print("\nPrice by year:")
print(df.groupby(df['date'].dt.year)['modal_price'].agg(['mean','std','min','max']).round(0))

Consecutive same-price rows (forward-filled): 392
That's 37.9% of all rows

Test period: 2025-07-03 to 2025-12-05
Train period: 2023-02-06 to 2025-01-28

Price by year:
        mean     std     min     max
date                                
2023  2792.0   983.0  1197.0  4253.0
2024  3165.0  1293.0  1540.0  5304.0
2025  1962.0   484.0  1356.0  4931.0


In [3]:
import sys
sys.path.append('..')
from src.data.database import get_raw_prices_df

raw_df = get_raw_prices_df(commodity='onion')
lasalgaon_markets = raw_df[raw_df['market'].str.contains('Lasalgaon', case=False)]['market'].unique()
print(lasalgaon_markets)

['Lasalgaon' 'Lasalgaon(Niphad)' 'Lasalgaon(Vinchur)']


In [2]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from src.utils.config import TRAIN_RATIO, VAL_RATIO

df = pd.read_csv('../data/processed/features_onion.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

n = len(df)
n_train = int(n * TRAIN_RATIO)
n_val   = int(n * VAL_RATIO)

train = df.iloc[:n_train]
val   = df.iloc[n_train:n_train+n_val]
test  = df.iloc[n_train+n_val:]

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"\nTrain price: mean=₹{train['modal_price'].mean():.0f} std=₹{train['modal_price'].std():.0f}")
print(f"Val price:   mean=₹{val['modal_price'].mean():.0f} std=₹{val['modal_price'].std():.0f}")
print(f"Test price:  mean=₹{test['modal_price'].mean():.0f} std=₹{test['modal_price'].std():.0f}")

# Check target_1d nulls per split
print(f"\nTarget nulls — Train: {train['target_1d'].isnull().sum()} | Val: {val['target_1d'].isnull().sum()} | Test: {test['target_1d'].isnull().sum()}")

print(f"\nTest dates: {test['date'].min().date()} to {test['date'].max().date()}")

Train: 448 | Val: 96 | Test: 97

Train price: mean=₹2580 std=₹1044
Val price:   mean=₹3524 std=₹1161
Test price:  mean=₹2089 std=₹455

Target nulls — Train: 22 | Val: 10 | Test: 12

Test dates: 2025-03-14 to 2025-12-05


In [3]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/features_onion.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

# Check price by year-quarter to understand the regime
df['yearq'] = df['date'].dt.to_period('Q')
print(df.groupby('yearq')['modal_price'].agg(['mean','std','count']).round(0))

          mean     std  count
yearq                        
2023Q1  2551.0  1089.0     12
2023Q2  1908.0   932.0     37
2023Q3  2060.0   581.0     77
2023Q4  2891.0   832.0     77
2024Q1  2274.0   930.0     88
2024Q2  2341.0  1034.0     71
2024Q3  3570.0   928.0     86
2024Q4  3889.0  1164.0     67
2025Q1  2584.0   472.0     47
2025Q2  1870.0   336.0     50
2025Q3  2266.0   559.0     15
2025Q4  2247.0   573.0     14


In [4]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from src.models.evaluate import evaluate_all

df = pd.read_csv('../data/processed/features_onion.csv', parse_dates=['date'])
df = df.sort_values('date').reset_index(drop=True)

series = df.set_index('date')['modal_price'].asfreq('D').ffill()

train_val = series[series.index < '2025-01-01']
test      = series[series.index >= '2025-01-01']

print(f"Train+Val: {len(train_val)} | Test: {len(test)}")
print(f"Test period: {test.index.min().date()} to {test.index.max().date()}")

# Fit on full train_val
model = SARIMAX(train_val, order=(1,1,1), seasonal_order=(1,1,1,7),
                enforce_stationarity=False, enforce_invertibility=False)
fit = model.fit(disp=False)

# One-step-ahead rolling forecast on test
history = list(train_val)
preds = []
for actual in test.values:
    m = SARIMAX(history, order=(1,1,1), seasonal_order=(1,1,1,7),
                enforce_stationarity=False, enforce_invertibility=False)
    f = m.fit(disp=False, start_params=fit.params)
    preds.append(f.forecast(1)[0])
    history.append(actual)

metrics = evaluate_all(test.values, np.array(preds), model_name='SARIMA')
print(f"\nSARIMA on 2025 test: RMSE={metrics['rmse']:.1f} MAE={metrics['mae']:.1f} MAPE={metrics['mape']:.2f}% R2={metrics['r2']:.3f}")

Train+Val: 694 | Test: 339
Test period: 2025-01-01 to 2025-12-05


c:\Users\Jeya Devi\OneDrive\Desktop\dev-projects\agri-price-forecast\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


2026-05-16 19:19:43 | INFO     | src.models.evaluate | [SARIMA] RMSE=300.74 | MAE=150.99 | MAPE=7.03% | SMAPE=6.93% | R2=0.6127

SARIMA on 2025 test: RMSE=300.7 MAE=151.0 MAPE=7.03% R2=0.613
